# Silver Layer — Market Observations

## Purpose
The Silver layer transforms Bronze cryptocurrency market observations
into validated, analysis-ready records.

Invalid observations are stored in the Silver quarantine table.

## Target Table
`workspace.default.silver_market_observation`

## Business Key
`coin_id` + `observation_timestamp`

## Data Dictionary

| Column | Data Type | Description | Key |
|---|---|---|---|
| coin_id | STRING | Cryptocurrency identifier | Composite key |
| observation_timestamp | TIMESTAMP | Market observation date and time | Composite key |
| price_usd | DOUBLE | Cryptocurrency price in USD | No |
| market_cap_usd | DOUBLE | Market capitalization in USD | No |
| total_volume_usd | DOUBLE | Trading volume in USD | No |
| source_file | STRING | Source file associated with the observation | No |
| load_timestamp | TIMESTAMP | Original Bronze load timestamp | No |

## Data Quality Rules
- Reject missing or blank cryptocurrency identifiers.
- Reject missing observation timestamps.
- Reject null, NaN, or negative market values.
- Store rejected observations with rejection reasons.
- Check duplicate business keys before Delta MERGE.

## Loading Strategy
Delta MERGE uses `coin_id` and `observation_timestamp` to update
existing observations and insert new ones without duplicating keys.

In [0]:
# Step 112: Silver pipeline setup and parameters

import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType,
    LongType
)
from delta.tables import DeltaTable

# Create execution parameters
dbutils.widgets.dropdown(
    "load_type",
    "full_load",
    ["full_load", "incremental_load"]
)

dbutils.widgets.text("processing_date", "")
dbutils.widgets.text("source_folder", "")

# Read parameter values
silver_load_type = dbutils.widgets.get("load_type")
silver_processing_date = dbutils.widgets.get("processing_date").strip()
silver_source_folder = dbutils.widgets.get("source_folder").strip()

# Capture the actual pipeline start time
silver_run_id = str(uuid.uuid4())
silver_start_time = datetime.now(timezone.utc).replace(tzinfo=None)

# Define table names
bronze_table_name = "workspace.default.bronze_coin_market_raw"
silver_table_name = "workspace.default.silver_market_observation"
quarantine_table_name = "workspace.default.silver_quarantine"
execution_log_table_name = "workspace.default.pipeline_execution_logs"

print("Silver pipeline initialized successfully!")
print("Run ID:", silver_run_id)
print("Load type:", silver_load_type)
print("Processing date:", silver_processing_date or "All dates")
print("Source folder:", silver_source_folder or "All Bronze sources")
print("Start time (UTC):", silver_start_time)

In [0]:
# Step 113: Define the explicit Silver schema

silver_schema = StructType([
    StructField("coin_id", StringType(), False),
    StructField("observation_timestamp", TimestampType(), False),
    StructField("price_usd", DoubleType(), True),
    StructField("market_cap_usd", DoubleType(), True),
    StructField("total_volume_usd", DoubleType(), True),
    StructField("source_file", StringType(), True),
    StructField("load_timestamp", TimestampType(), True)
])

# Business key: coin_id + observation_timestamp

print("Silver schema defined successfully!")

for field in silver_schema.fields:
    print(
        f"{field.name}: {field.dataType.simpleString()} "
        f"(nullable={field.nullable})"
    )

print("Silver business key: coin_id + observation_timestamp")

In [0]:
# Step 114: Read parameter-selected Bronze records

# Read the Bronze Delta table
bronze_df = spark.table(bronze_table_name)

# Validate the optional processing date
if silver_processing_date:
    from datetime import date

    try:
        date.fromisoformat(silver_processing_date)
    except ValueError:
        raise ValueError(
            "processing_date must use YYYY-MM-DD format."
        )

# Filter by observation date if provided
if silver_processing_date:
    bronze_df = bronze_df.filter(
        F.to_date(F.col("observation_timestamp")) ==
        F.lit(silver_processing_date).cast("date")
    )

# Filter by source folder if provided
if silver_source_folder:
    bronze_df = bronze_df.filter(
        F.col("source_file").contains(silver_source_folder)
    )

# Count selected records
silver_rows_read = bronze_df.count()

print("Bronze records selected:", silver_rows_read)
print("Load type:", silver_load_type)
print("Processing date:", silver_processing_date or "All dates")
print("Source folder:", silver_source_folder or "All Bronze sources")

In [0]:
# Step 115: Silver data-quality validation

silver_invalid_condition = (
    F.col("coin_id").isNull()
    | (F.trim(F.col("coin_id")) == "")
    | F.col("observation_timestamp").isNull()
    | F.col("price_usd").isNull()
    | F.isnan("price_usd")
    | (F.col("price_usd") < 0)
    | F.col("market_cap_usd").isNull()
    | F.isnan("market_cap_usd")
    | (F.col("market_cap_usd") < 0)
    | F.col("total_volume_usd").isNull()
    | F.isnan("total_volume_usd")
    | (F.col("total_volume_usd") < 0)
)

# Separate valid and invalid observations
silver_valid_df = bronze_df.filter(
    ~silver_invalid_condition
)

silver_invalid_df = bronze_df.filter(
    silver_invalid_condition
)

# Count records
silver_valid_count = silver_valid_df.count()
silver_invalid_count = silver_invalid_df.count()

print("Total Bronze records:", silver_rows_read)
print("Valid Silver records:", silver_valid_count)
print("Invalid Silver records:", silver_invalid_count)

# Verify that no records were lost
if silver_valid_count + silver_invalid_count != silver_rows_read:
    raise ValueError(
        "Data-quality validation failed: record counts do not match."
    )

print("Silver data-quality validation successful!")

In [0]:
# Step 116: Generate rejection reasons for invalid observations

silver_invalid_details_df = (
    silver_invalid_df
    .withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("coin_id").isNull() |
                (F.trim(F.col("coin_id")) == ""),
                "Invalid coin ID"
            ),
            F.when(
                F.col("observation_timestamp").isNull(),
                "Invalid observation timestamp"
            ),
            F.when(
                F.col("price_usd").isNull() |
                F.isnan("price_usd") |
                (F.col("price_usd") < 0),
                "Invalid price"
            ),
            F.when(
                F.col("market_cap_usd").isNull() |
                F.isnan("market_cap_usd") |
                (F.col("market_cap_usd") < 0),
                "Invalid market cap"
            ),
            F.when(
                F.col("total_volume_usd").isNull() |
                F.isnan("total_volume_usd") |
                (F.col("total_volume_usd") < 0),
                "Invalid total volume"
            )
        )
    )
)

print(
    "Invalid observations with rejection reasons:",
    silver_invalid_details_df.count()
)

display(
    silver_invalid_details_df.select(
        "coin_id",
        "market_cap_usd",
        "rejection_reason"
    )
)

In [0]:
# Step 117: Save invalid observations to Silver quarantine

silver_quarantine_source_df = silver_invalid_details_df.select(
    "coin_id",
    "source_file",
    "timestamp_ms",
    "observation_timestamp",
    "price_usd",
    "market_cap_usd",
    "total_volume_usd",
    "rejection_reason",
    "load_timestamp"
)

# Connect to the existing quarantine table
quarantine_delta = DeltaTable.forName(
    spark,
    quarantine_table_name
)

# Use null-safe comparisons for business keys
(
    quarantine_delta.alias("target")
    .merge(
        silver_quarantine_source_df.alias("source"),
        """
        target.coin_id <=> source.coin_id
        AND target.observation_timestamp <=> source.observation_timestamp
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver quarantine MERGE completed successfully!")

print(
    "Invalid observations processed:",
    silver_quarantine_source_df.count()
)

print(
    "Total quarantine records:",
    spark.table(quarantine_table_name).count()
)

In [0]:
# Step 118: Prepare valid records for Silver MERGE

silver_source_df = silver_valid_df.select(
    F.col("coin_id").cast("string").alias("coin_id"),
    F.col("observation_timestamp").cast("timestamp"),
    F.col("price_usd").cast("double"),
    F.col("market_cap_usd").cast("double"),
    F.col("total_volume_usd").cast("double"),
    F.col("source_file").cast("string"),
    F.col("load_timestamp").cast("timestamp")
)

# Check duplicate business keys
silver_duplicate_count = (
    silver_source_df
    .groupBy("coin_id", "observation_timestamp")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

silver_prepared_count = silver_source_df.count()

print("Valid Silver records prepared:", silver_prepared_count)
print("Duplicate business keys:", silver_duplicate_count)

if silver_duplicate_count > 0:
    raise ValueError(
        "Silver MERGE stopped: duplicate business keys detected."
    )

print("Silver records ready for MERGE!")

In [0]:
# Step 119: MERGE valid observations into Silver

# Count records before MERGE
silver_before_count = spark.table(silver_table_name).count()

# Connect to Silver Delta table
silver_delta = DeltaTable.forName(
    spark,
    silver_table_name
)

# Perform idempotent MERGE
(
    silver_delta.alias("target")
    .merge(
        silver_source_df.alias("source"),
        """
        target.coin_id = source.coin_id
        AND target.observation_timestamp = source.observation_timestamp
        """
    )
    .whenMatchedUpdate(set={
        "price_usd": "source.price_usd",
        "market_cap_usd": "source.market_cap_usd",
        "total_volume_usd": "source.total_volume_usd",
        "source_file": "source.source_file",
        "load_timestamp": "source.load_timestamp"
    })
    .whenNotMatchedInsertAll()
    .execute()
)

# Read the actual MERGE metrics from Delta history
silver_latest_merge = (
    spark.sql(f"DESCRIBE HISTORY {silver_table_name}")
    .filter(F.col("operation") == "MERGE")
    .orderBy(F.col("version").desc())
    .first()
)

silver_merge_metrics = silver_latest_merge["operationMetrics"]

silver_rows_inserted = int(
    silver_merge_metrics.get("numTargetRowsInserted", 0)
)

silver_rows_updated = int(
    silver_merge_metrics.get("numTargetRowsUpdated", 0)
)

silver_after_count = spark.table(silver_table_name).count()

print("Silver MERGE completed successfully!")
print("Silver records before:", silver_before_count)
print("Silver records after:", silver_after_count)
print("Rows inserted:", silver_rows_inserted)
print("Rows updated:", silver_rows_updated)

In [0]:
# Step 120: Save final Silver execution log

from datetime import datetime, timezone

# Capture pipeline end time
silver_end_time = datetime.now(timezone.utc).replace(tzinfo=None)

# Match the existing execution log table schema
execution_log_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("processing_date", StringType(), True),
    StructField("source_folder", StringType(), True),
    StructField("start_timestamp", TimestampType(), True),
    StructField("end_timestamp", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("rows_quarantined", LongType(), True),
    StructField("error_message", StringType(), True)
])

# Prepare one execution log record
silver_log_data = [(
    silver_run_id,
    "silver",
    silver_table_name,
    silver_load_type,
    silver_processing_date or None,
    silver_source_folder or None,
    silver_start_time,
    silver_end_time,
    "SUCCESS",
    silver_rows_read,
    silver_rows_inserted,
    silver_rows_updated,
    silver_invalid_count,
    None
)]

silver_log_df = spark.createDataFrame(
    silver_log_data,
    schema=execution_log_schema
)

# Avoid inserting the same run ID twice
from delta.tables import DeltaTable

execution_log_delta = DeltaTable.forName(
    spark,
    execution_log_table_name
)

(
    execution_log_delta.alias("target")
    .merge(
        silver_log_df.alias("source"),
        "target.run_id = source.run_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Final Silver execution log saved successfully!")
print("Run ID:", silver_run_id)
print("Status: SUCCESS")
print("Rows read:", silver_rows_read)
print("Rows inserted:", silver_rows_inserted)
print("Rows updated:", silver_rows_updated)
print("Rows quarantined:", silver_invalid_count)

display(
    spark.table(execution_log_table_name)
    .filter(F.col("run_id") == silver_run_id)
)